# Module 5 — Approved Retrieval and Citations

This notebook uses a fictional corpus and an offline extractive generator. From `module-05/`, install `python -m pip install -e ".[dev]"`. No API key is needed. Work through Build → Break → Measure → Improve. The date is pinned to 29 September 2026 so the results match the article.

## 1. Build: load, chunk and retrieve

The loader rejects a bad file by name. Chunks are split on `##` headings, and each ID is a hash of the document, version, position and text.

In [ ]:
from dataclasses import replace
from rag_lab.assistant import GroundedAssistant
from rag_lab.corpus import ingest, load_corpus, parse_document
from rag_lab.grounding import ExtractiveGenerator
from rag_lab.retrieval import Retriever
from rag_lab.run_eval import EVAL_DATE, evaluate, load_cases, summary

docs = load_corpus()
chunks = ingest(docs)
retriever = Retriever(chunks)
assistant = GroundedAssistant(retriever, ExtractiveGenerator())
print(len(docs), "document versions,", len(chunks), "chunks")

In [ ]:
def ask(message, region=""):
    result = assistant.answer(message, region=region, today=EVAL_DATE)
    print(f"[{result.route} · {result.grounding}] {result.response}")
    for s in result.sources:
        print(f"  [{s['n']}] {s['doc_id']} v{s['version']} — {s['heading']} ({s['chunk_id']})")
    return result

ask("Can I return an opened item?")
ask("Is cash on delivery available?")
ask("Is cash on delivery available?", region="PK")
ask("Do you price match?")
ask("Cancel my order ORD-1001.");

## 2. Break: metadata, dates and versions

Exercise 1. Break a copy of a document in each way below and confirm the error names the file.

In [ ]:
raw = (docs[0].source, open(f"../data/corpus/{docs[0].source}", encoding="utf-8").read())
for broken in (raw[1].replace("status: approved", "status: published"),
               raw[1].replace("version: ", "version: one"),
               raw[1].split("---", 2)[2]):
    try:
        parse_document(broken, raw[0])
    except ValueError as exc:
        print(exc)

In [ ]:
# The same question on two dates: RET-30 v3 takes effect on 2027-01-01.
for today in (EVAL_DATE, "2027-02-01"):
    result = retriever.retrieve("Is the return period 45 days now?", today=today)
    print(today, [(c.doc_id, c.version) for c in result.passages], result.excluded)

## 3. Measure: the frozen evaluation

Exercise 2. Sweep `min_score` and read which cases move, not just the totals.

In [ ]:
cases = load_cases()
print(summary(evaluate(retriever, ExtractiveGenerator(), cases)))

In [ ]:
for value in (2.0, 3.0, 3.5, 4.0, 5.0):
    result = evaluate(Retriever(chunks, min_score=value), ExtractiveGenerator(), cases)
    print(value, f"recall {result['recall_hits']}/{result['recall_total']}",
          f"routes {result['routes']}/{result['total']}", [f["id"] for f in result["failures"]])

## 4. Improve

Exercise 3. RAG-008 ("What is the delivery charge?" with no region) answers from the billing policy. Try one fix: a content fix (edit a copy of the corpus), a routing fix (ask for a region before retrieving region-dependent topics), or a generation fix (a live model that declines). Re-run section 3 and check nothing else moved.

Exercise 4. RAG-019 ("Do gift cards expire?") passes every check but does not answer the question. Read the answer below, then write down how you would catch this without reading every answer.

In [ ]:
ask("What is the delivery charge?")
ask("Do gift cards expire?");

## 5. Optional live mode

With `OPENAI_API_KEY` and `OPENAI_MODEL` in `module-05/.env` and `pip install -e ".[dev,live]"`, run `python -m rag_lab.run_eval --live --output reports/module-05-live-results.json`. Live calls cost money and use synthetic cases only.